# MCAM Server

Loads the multi-collection ChromaDB from Hugging Face, embeds query images, retrieves + reranks AAT keywords, and serves everything via a FastAPI endpoint.

**New features:**
- Swap collection / embedding model in one config cell
- Filter results by facet or hierarchy
- `/collections` and `/facets` discovery endpoints

In [1]:
# %%capture

# Install llama-cpp-python from prebuilt CUDA wheel (skips the long compile)
# Adjust cu124 to match your CUDA toolkit version if needed (check with: !nvcc --version)
!pip install llama-cpp-python --prefer-binary --extra-index-url=https://abetlen.github.io/llama-cpp-python/whl/cu124

!pip install torch transformers chromadb datasets qwen-vl-utils hf_xet langchain-chroma langchain-core Pillow huggingface-hub python-dotenv fastapi uvicorn pyngrok nest-asyncio python-multipart sentencepiece open_clip_torch

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 GB 436.9 kB/s eta 0:00:0000:01:09mm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 80.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 100.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.8 MB/s eta 0:00:00
   

Logged in to Hugging Face
NGROK_TOKEN loaded


# (Optional) Flash Attention

Requires Ampere or newer GPU. **T4 does NOT support Flash Attention** — skip this cell on T4.

In [3]:
!nvidia-smi

# Uncomment below ONLY on Ampere+ GPU (L4, A100, etc.)
# !pip install ninja
# !pip install flash-attn --no-build-isolation

Thu Apr 16 11:48:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Configuration

Change `COLLECTION_NAME` and `EMBEDDING_TYPE` to switch which collection and model the server uses. They must match — e.g. a collection embedded with Qwen needs `EMBEDDING_TYPE = "qwen"` so query images are embedded in the same space.

In [4]:
# ── Active collection ──
COLLECTION_NAME = "aat_terms_qwen2b"
EMBEDDING_TYPE  = "qwen"   # must match what was used to embed the collection

# ── Model IDs (must match the embedding notebook) ──
EMBEDDING_MODELS = {
    "qwen":     {"model_id": "Qwen/Qwen3-VL-Embedding-2B"},
    "openclip": {"model_id": "ViT-SO400M-14-SigLIP-384", "pretrained": "webli"},
}

# ── HF dataset repo containing the VDB ──
VDB_REPO = "KeeganCarey/mcam-vdb"

# ── Keyword generation defaults ──
TERM_COUNT  = 20
LAMBDA_MULT = 0.96   # MMR: 1.0 = pure relevance, 0.0 = pure diversity

# ── Frontend ──
FORCE_REBUILD_FRONTEND = True  # Set to False to skip rebuild when dist/ already exists
!rm -rf /content/Mills-Museum/src/frontend/web/dist


# Setup Paths

In [5]:
import os, sys, subprocess
from pathlib import Path
from huggingface_hub import snapshot_download

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

# ── Change this to match the branch you're working on ──
BRANCH = "cleanup/git-rebase"

if IN_COLAB:
    repo_path = "/content/Mills-Museum"
    if not os.path.exists(repo_path):
        subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/cs2535-oakhoury-2026spring/Mills-Museum.git", repo_path])
    else:
        subprocess.run(["git", "fetch", "origin"], cwd=repo_path)
        subprocess.run(["git", "checkout", BRANCH], cwd=repo_path)
        subprocess.run(["git", "pull", "origin", BRANCH], cwd=repo_path)
else:
    repo_path = str(Path(os.getcwd()).parent) if Path(os.getcwd()).name == "colab" else os.getcwd()
    if not os.path.exists(os.path.join(repo_path, "src")):
        repo_path = os.getcwd()

# Build the frontend (Vite → dist/)
frontend_dir = os.path.join(repo_path, "src", "frontend", "web")
dist_dir = os.path.join(frontend_dir, "dist")
if not os.path.isdir(dist_dir):
    print("Building frontend...")
    subprocess.run(["npm", "install"], cwd=frontend_dir, check=True)
    subprocess.run(["npx", "vite", "build"], cwd=frontend_dir, check=True)
    print("Frontend built!")
else:
    print("Frontend dist/ already exists, skipping build")

VDB_PATH = snapshot_download(repo_id=VDB_REPO, repo_type="dataset")

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Branch:      {BRANCH}")
print(f"Repo path:   {repo_path}")
print(f"VDB path:    {VDB_PATH}")

Building frontend...
Frontend built!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Environment: Colab
Branch:      cleanup/git-rebase
Repo path:   /content/Mills-Museum
VDB path:    /root/.cache/huggingface/hub/datasets--KeeganCarey--mcam-vdb/snapshots/5b357eff536ffd09b2bae6a5cb4bb8855c872be9


# Load Vector Store

Opens the ChromaDB VDB, loads the configured collection, and pre-computes the list of available facets & hierarchies for the `/facets` endpoint.

In [6]:
import chromadb
from langchain_chroma import Chroma

# Raw client — used for listing collections and facet discovery
_chroma_client = chromadb.PersistentClient(path=VDB_PATH)
available_collections = [c.name for c in _chroma_client.list_collections()]
print(f"Available collections: {available_collections}")

assert COLLECTION_NAME in available_collections, (
    f"Collection '{COLLECTION_NAME}' not found. Available: {available_collections}"
)

# LangChain wrapper — gives us MMR search
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=VDB_PATH,
    collection_metadata={"hnsw:space": "cosine"},
)
print(f"Loaded '{COLLECTION_NAME}' — {vectorstore._collection.count()} documents")

# Pre-compute facets and hierarchies from metadata
_all_meta = vectorstore._collection.get(include=["metadatas"])
AVAILABLE_FACETS = sorted(set(
    m.get("facet", "") for m in _all_meta["metadatas"] if m.get("facet")
))
AVAILABLE_HIERARCHIES = sorted(set(
    m.get("hierarchy", "") for m in _all_meta["metadatas"] if m.get("hierarchy")
))
del _all_meta

print(f"Facets ({len(AVAILABLE_FACETS)}): {AVAILABLE_FACETS}")
print(f"Hierarchies ({len(AVAILABLE_HIERARCHIES)}): {AVAILABLE_HIERARCHIES}")

Available collections: ['aat_terms_qwen2b', 'aat_terms_siglip']
Loaded 'aat_terms_qwen2b' — 19888 documents
Facets (12): ['D.DG', 'D.DL', 'F.FL', 'H.HG', 'H.HL', 'K.KM', 'K.KQ', 'K.KT', 'M.MT', 'V.PJ', 'V.R', 'V.T']
Hierarchies (12): ['Built Environment', 'Color', 'Components', 'Design Elements', 'Events', 'Furnishings and Equipment', 'Living Organisms', 'Materials', 'People', 'Physical and Mental Activities', 'Processes and Techniques', 'Styles and Periods']


# Load Embedding Model

Loads the embedding model that matches `EMBEDDING_TYPE`. This is used at query time to embed the uploaded image into the same vector space as the stored terms.

In [7]:
import torch
import sys
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

try:
    import flash_attn
    ATTN_IMPL = "flash_attention_2"
    print("Flash Attention available")
except ImportError:
    ATTN_IMPL = "sdpa"
    print("Using SDPA attention")

embed_cfg = EMBEDDING_MODELS[EMBEDDING_TYPE]
model_id  = embed_cfg["model_id"]

if EMBEDDING_TYPE == "qwen":
    from huggingface_hub import hf_hub_download
    import importlib.util

    _script = hf_hub_download(repo_id=model_id, filename="scripts/qwen3_vl_embedding.py")
    _spec = importlib.util.spec_from_file_location("qwen3_vl_embedding", _script)
    _mod = importlib.util.module_from_spec(_spec)
    sys.modules["qwen3_vl_embedding"] = _mod
    _spec.loader.exec_module(_mod)

    embedding_model = _mod.Qwen3VLEmbedder(
        model_name_or_path=model_id,
        dtype=torch.float16 if device == "cuda" else torch.float32,
        attn_implementation=ATTN_IMPL,
    )

    def embed_image(image):
        features = embedding_model.process([{"image": image, "text": ""}])
        features = torch.nn.functional.normalize(features, p=2, dim=1)
        return features.cpu().float().squeeze().tolist()

elif EMBEDDING_TYPE == "openclip":
    import open_clip

    pretrained = embed_cfg.get("pretrained", "webli")
    _oc_model, _, _oc_preprocess = open_clip.create_model_and_transforms(
        model_id, pretrained=pretrained, device=device,
    )
    _oc_model.eval()

    def embed_image(image):
        img_tensor = _oc_preprocess(image).unsqueeze(0).to(device)
        with torch.no_grad(), torch.amp.autocast(device):
            features = _oc_model.encode_image(img_tensor)
            features = torch.nn.functional.normalize(features, p=2, dim=1)
        return features.cpu().float().squeeze().tolist()

else:
    raise ValueError(f"Unknown EMBEDDING_TYPE: {EMBEDDING_TYPE}")

print(f"Loaded {EMBEDDING_TYPE} embedding model: {model_id}")

Device: cuda
Using SDPA attention


qwen3_vl_embedding.py: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

Loaded qwen embedding model: Qwen/Qwen3-VL-Embedding-2B


# Load Reranker

Downloads and loads the Qwen3-VL-Reranker-2B as a GGUF (Q6_K imatrix quant) via llama-cpp-python. Used to re-score candidates after the initial vector retrieval.

In [8]:
import math
import base64
import io
from tqdm import tqdm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

print("Downloading Qwen3-VL-Reranker-2B GGUF...")

reranker_gguf_path = hf_hub_download(
    repo_id="mradermacher/Qwen3-VL-Reranker-2B-i1-GGUF",
    filename="Qwen3-VL-Reranker-2B.i1-Q6_K.gguf",
)
reranker_mmproj_path = hf_hub_download(
    repo_id="mradermacher/Qwen3-VL-Reranker-2B-GGUF",
    filename="Qwen3-VL-Reranker-2B.mmproj-Q8_0.gguf",
)

reranker_llm = Llama(
    model_path=reranker_gguf_path,
    clip_model_path=reranker_mmproj_path,
    n_ctx=4096,
    n_gpu_layers=-1,
    logits_all=True,
    verbose=False,
)

YES_TOKEN = reranker_llm.tokenize(b"yes", add_bos=False)[0]
NO_TOKEN  = reranker_llm.tokenize(b"no",  add_bos=False)[0]
print(f"Reranker ready — yes={YES_TOKEN}, no={NO_TOKEN}")


def _pil_to_data_uri(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f"data:image/png;base64,{b64}"


def rerank_with_gguf(instruction: str, query: dict, documents: list[dict]) -> list[float]:
    img_uri = _pil_to_data_uri(query["image"]) if query.get("image") else None
    query_text = query.get("text", "")
    scores = []

    for doc in tqdm(documents, desc="Reranking"):
        user_content = []
        if img_uri:
            user_content.append({"type": "image_url", "image_url": {"url": img_uri}})
        user_content.append({
            "type": "text",
            "text": (
                f"Instruction: {instruction}\n"
                f"Query: {query_text}\n"
                f"Document: {doc['text']}\n"
                "Is the Document relevant to the Query? Answer only 'yes' or 'no'."
            ),
        })

        messages = [
            {"role": "system", "content": "Judge whether the Document is relevant to the Query. Answer only 'yes' or 'no'."},
            {"role": "user", "content": user_content},
        ]

        resp = reranker_llm.create_chat_completion(
            messages=messages,
            max_tokens=1,
            logprobs=True,
            top_logprobs=10,
            temperature=0.0,
        )

        top_lps = resp["choices"][0]["logprobs"]["content"][0]["top_logprobs"]
        lp_map = {entry["token"].strip().lower(): entry["logprob"] for entry in top_lps}
        lp_yes = lp_map.get("yes", -100.0)
        lp_no  = lp_map.get("no",  -100.0)
        scores.append(math.exp(lp_yes) / (math.exp(lp_yes) + math.exp(lp_no)))

    return scores


print("Reranker loaded!")

Qwen3-VL-Reranker-2B.i1-Q6_K.gguf:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Qwen3-VL-Reranker-2B.mmproj-Q8_0.gguf:   0%|          | 0.00/445M [00:00<?, ?B/s]

llama_context: n_ctx_seq (4096) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


Reranker ready — yes=9693, no=2152
Reranker loaded!


# API Server

Endpoints:
- `POST /predict` — upload an image, get initial (unscored) AAT keywords + a `job_id`. Reranking runs in the background.
- `GET /predict-status/{job_id}` — poll for reranking progress. Returns scored keywords as they complete.
- `GET /collections` — list all collections in the VDB.
- `GET /facets` — list available facets and hierarchies for filtering.
- `GET /` — serve the frontend.

In [ ]:
import uvicorn
import nest_asyncio
import threading
import uuid
import time
import math
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.staticfiles import StaticFiles
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok, conf
from PIL import Image
from typing import Optional
import io
import os

nest_asyncio.apply()
conf.get_default().auth_token = NGROK_TOKEN

# ── Job store for progressive reranking ──
_jobs = {}
_jobs_lock = threading.Lock()
_reranker_lock = threading.Lock()
JOB_TTL_SECONDS = 600


def _sweep_expired_jobs():
    """Remove jobs older than JOB_TTL_SECONDS."""
    now = time.time()
    with _jobs_lock:
        expired = [jid for jid, j in _jobs.items() if now - j["created_at"] > JOB_TTL_SECONDS]
        for jid in expired:
            del _jobs[jid]


def _rerank_job(job_id, image, input_docs, labels, doc_metadata):
    """Background thread: reranks documents one-by-one, updating job state after each."""
    try:
        img_uri = _pil_to_data_uri(image)
        instruction = "Retrieve Art & Architecture Thesaurus terms relevant to the given image."

        with _reranker_lock:
            for i, doc in enumerate(input_docs):
                user_content = []
                if img_uri:
                    user_content.append({"type": "image_url", "image_url": {"url": img_uri}})
                user_content.append({
                    "type": "text",
                    "text": (
                        f"Instruction: {instruction}\n"
                        f"Query: \n"
                        f"Document: {doc['text']}\n"
                        "Is the Document relevant to the Query? Answer only 'yes' or 'no'."
                    ),
                })

                messages = [
                    {"role": "system", "content": "Judge whether the Document is relevant to the Query. Answer only 'yes' or 'no'."},
                    {"role": "user", "content": user_content},
                ]

                resp = reranker_llm.create_chat_completion(
                    messages=messages,
                    max_tokens=1,
                    logprobs=True,
                    top_logprobs=10,
                    temperature=0.0,
                )

                top_lps = resp["choices"][0]["logprobs"]["content"][0]["top_logprobs"]
                lp_map = {entry["token"].strip().lower(): entry["logprob"] for entry in top_lps}
                lp_yes = lp_map.get("yes", -100.0)
                lp_no  = lp_map.get("no",  -100.0)
                score = math.exp(lp_yes) / (math.exp(lp_yes) + math.exp(lp_no))

                with _jobs_lock:
                    job = _jobs.get(job_id)
                    if job is None:
                        return
                    job["keywords"][i]["score"] = round(float(score) * 100, 1)
                    job["completed"] = i + 1

        # Done — apply final sort by score descending
        with _jobs_lock:
            job = _jobs.get(job_id)
            if job:
                job["status"] = "done"
                job["keywords"].sort(
                    key=lambda k: k["score"] if k["score"] is not None else -1,
                    reverse=True,
                )

    except Exception as e:
        with _jobs_lock:
            job = _jobs.get(job_id)
            if job:
                job["status"] = "error"
                job["error"] = str(e)


app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Serve the Vite production build
DIST_PATH = os.path.join(repo_path, "src", "frontend", "web", "dist")


@app.get("/collections")
async def list_collections():
    """List all collections in the VDB."""
    return {
        "active": COLLECTION_NAME,
        "available": available_collections,
    }


@app.get("/facets")
async def list_facets():
    """List facets and hierarchies available for filtering."""
    return {
        "facets": AVAILABLE_FACETS,
        "hierarchies": AVAILABLE_HIERARCHIES,
    }


@app.post("/predict")
async def predict(
    file: UploadFile = File(...),
    term_count: int = Form(default=TERM_COUNT),
    facets: Optional[str] = Form(default=None),
    hierarchies: Optional[str] = Form(default=None),
):
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    # Embed query image
    query_embedding = embed_image(image)

    # Build optional where-filter for facets / hierarchies
    conditions = []
    if facets:
        facet_list = [f.strip() for f in facets.split(",")]
        conditions.append({"facet": {"$in": facet_list}})
    if hierarchies:
        hierarchy_list = [h.strip() for h in hierarchies.split(",")]
        conditions.append({"hierarchy": {"$in": hierarchy_list}})

    where_filter = None
    if len(conditions) == 1:
        where_filter = conditions[0]
    elif len(conditions) > 1:
        where_filter = {"$and": conditions}

    # MMR retrieval (fast)
    mmr_docs = vectorstore.max_marginal_relevance_search_by_vector(
        embedding=query_embedding,
        k=term_count,
        fetch_k=term_count * 4,
        lambda_mult=LAMBDA_MULT,
        filter=where_filter,
    )

    labels = []
    input_docs = []
    doc_metadata = []
    for doc in mmr_docs:
        term = doc.metadata.get("preferred_term", doc.page_content.split(":")[0])
        labels.append(term)
        input_docs.append({"text": doc.page_content})
        doc_metadata.append(doc.metadata)

    # Create job with unscored keywords
    job_id = str(uuid.uuid4())
    initial_keywords = [
        {
            "label":      label,
            "score":      None,
            "facet":      meta.get("facet", ""),
            "hierarchy":  meta.get("hierarchy", ""),
            "subject_id": meta.get("subject_id", ""),
        }
        for label, meta in zip(labels, doc_metadata)
    ]

    with _jobs_lock:
        _jobs[job_id] = {
            "status": "reranking",
            "total": len(input_docs),
            "completed": 0,
            "keywords": initial_keywords,
            "error": None,
            "created_at": time.time(),
        }

    # Start background reranking
    thread = threading.Thread(
        target=_rerank_job,
        args=(job_id, image, input_docs, labels, doc_metadata),
        daemon=True,
    )
    thread.start()

    return {"job_id": job_id, "keywords": initial_keywords}


@app.get("/predict-status/{job_id}")
async def predict_status(job_id: str):
    """Poll for reranking progress. Returns current scores and completion count."""
    _sweep_expired_jobs()

    with _jobs_lock:
        job = _jobs.get(job_id)

    if not job:
        return JSONResponse(status_code=404, content={"error": "Job not found"})

    return {
        "status": job["status"],
        "total": job["total"],
        "completed": job["completed"],
        "keywords": job["keywords"],
        "error": job.get("error"),
    }


# Mount the built frontend as a catch-all (after API routes so they take priority)
app.mount("/", StaticFiles(directory=DIST_PATH, html=True), name="frontend")

# Launch
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")
print("Open this URL in your browser to use the app!")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [6000]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: NgrokTunnel: "https://f1d4-34-158-125-23.ngrok-free.app" -> "http://localhost:8000"
Open this URL in your browser to use the app!
INFO:     144.91.51.3:0 - "GET / HTTP/1.1" 200 OK
INFO:     144.91.51.3:0 - "GET /assets/index-BKA89EeN.js HTTP/1.1" 200 OK
INFO:     144.91.51.3:0 - "GET /assets/index-BRNVHChE.css HTTP/1.1" 200 OK
INFO:     144.91.51.3:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     144.91.51.3:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     144.91.51.3:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     144.91.51.3:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     144.91.51.3:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     144.91.51.3:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     144.91.51.3:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     144.91.51.3:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     144.91.51.3:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     144.91.51.3:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     144.91.51.3:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     1